# Variational Quantum Eigensolver (VQE) pour H₂

Principe variationnel:
$$E_0 = \min_{\theta} \langle \psi(\theta) | H | \psi(\theta) \rangle$$

Hamiltonien $\text{H}_2$ dans la base STO-3G: matrice $4 \times 4$.

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

### Hamiltonien de H₂

$$H = \sum_{pq} h_{pq} a_p^\dagger a_q + \frac12 \sum_{pqrs} g_{pqrs} a_p^\dagger a_q^\dagger a_r a_s$$

Transformé en opérateurs Pauli via Jordan-Wigner.

In [ ]:
import pennylane as qml

H2_coeffs = [
    (-0.0420798, 'IIII'),
    (0.1777129, 'ZIII'),
    (0.1777129, 'IZII'),
    (-0.242743, 'IIZI'),
    (-0.242743, 'IIIZ'),
    (0.170597, 'ZZII'),
    (0.0447501, 'ZIZI'),
    (0.0447501, 'ZIIZ'),
    (0.0447501, 'IZZI'),
    (0.0447501, 'IZIZ'),
    (0.170597, 'IIZZ'),
    (0.122933, 'XXXX'),
    (0.122933, 'XXYY'),
    (-0.122933, 'YXYX'),
    (-0.122933, 'YYXX'),
]

In [ ]:
H2_obs = []
for coeff, op_str in H2_coeffs:
    term = qml.Identity(0)
    for i, c in enumerate(op_str):
        if c == 'X':
            term = term @ qml.PauliX(i)
        elif c == 'Y':
            term = term @ qml.PauliY(i)
        elif c == 'Z':
            term = term @ qml.PauliZ(i)
        elif c == 'I':
            term = term @ qml.Identity(i)
    H2_obs.append(term)

H_coeffs = np.array([c for c, _ in H2_coeffs])
H_ham = qml.Hamiltonian(H_coeffs, H2_obs)

### Ansatz hardware-efficient

$$|\psi(\theta)\rangle = U_{\text{ent}}(\theta_2) U_{\text{rot}}(\theta_1) |00\rangle$$

Couches de rotations + portes CNOT.

In [ ]:
dev = qml.device('default.qubit', wires=4, shots=None)

@qml.qnode(dev)
def cost_fn(params):
    qml.BasisState(np.array([1, 1, 0, 0]), wires=[0, 1, 2, 3])
    for layer in range(2):
        for i in range(4):
            qml.RY(params[layer, i], wires=i)
        for i in range(3):
            qml.CNOT(wires=[i, i + 1])
    return qml.expval(H_ham)

In [ ]:
np.random.seed(42)
params = np.random.uniform(0, 2 * np.pi, (2, 4), requires_grad=True)
print('Énergie initiale:', cost_fn(params))

### Optimisation classique-quantique

Boucle d'optimisation avec descente de gradient (Adam).

In [ ]:
opt = qml.AdamOptimizer(stepsize=0.1)
n_steps = 100
energies = []

for step in range(n_steps):
    params = np.array(params, requires_grad=True)
    params, energy = opt.step_and_cost(cost_fn, params)
    energies.append(energy)
    if step % 10 == 0:
        print(f'Étape {step:3d}: E = {energy:.6f} Ha')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(energies, linewidth=2)
plt.axhline(-1.136, color='r', linestyle='--', label='E₀ FCI ≈ -1.136 Ha')
plt.xlabel('Itération')
plt.ylabel('Énergie (Ha)')
plt.title('Convergence VQE pour H₂')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(f'\nÉnergie finale VQE: {energies[-1]:.6f} Ha')
print(f'Énergie exacte FCI:  -1.136 Ha')
print(f'Erreur: {abs(energies[-1] - (-1.136)):.6f} Ha')

## Questions

**Q1.** Modifier l'ansatz en utilisant $U_{\text{CCSD}}$ tronqué (ansatz UCCSD avec 2 couches). Comparer la vitesse de convergence et l'énergie finale avec l'ansatz hardware-efficient.

**Q2.** Ajouter du bruit de mesure en utilisant `default.mixed` ou un appareil avec `qml.transforms.insert` (canal dépolarisant). Comment l'énergie finale et la convergence sont-elles affectées?

In [ ]:
# Q2: VQE avec bruit
dev_noisy = qml.device('default.mixed', wires=4, shots=1000)

@qml.qnode(dev_noisy)
def cost_fn_noisy(params):
    qml.BasisState(np.array([1, 1, 0, 0]), wires=[0, 1, 2, 3])
    for layer in range(2):
        for i in range(4):
            qml.RY(params[layer, i], wires=i)
        for i in range(3):
            qml.CNOT(wires=[i, i + 1])
            qml.DepolarizingChannel(0.01, wires=i)
    return qml.expval(H_ham)

params_noisy = np.random.uniform(0, 2 * np.pi, (2, 4), requires_grad=True)
# Optimisation avec bruit
energies_noisy = []
for step in range(50):
    params_noisy, e = opt.step_and_cost(cost_fn_noisy, params_noisy)
    energies_noisy.append(e)

plt.plot(energies_noisy, label='Avec bruit', linewidth=2)
plt.plot(energies, label='Sans bruit', linewidth=2)
plt.xlabel('Itération')
plt.ylabel('Énergie (Ha)')
plt.title('VQE: comparaison bruité vs idéal')
plt.legend()
plt.grid(True)
plt.show()